<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Why did they call?

**[Why did they call?](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/)** · Speech AI: Transcribe, Understand and Speak · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the reason for each call with keyword rules, with a text classifier and with a recorded LLM answer, compare them on the same test calls, and measure how much each one loses when it reads a recognised transcript instead of the reference.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the LLM answers are recorded. |
| **You should know** | Accuracy and test sets, from the [machine learning course](https://learning.nextia-ai.com/courses/ml/), and the recorded transcripts, from [Whisper sizes on a laptop](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M06-L01-why-did-they-call/why-did-they-call-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, scikit-learn 1.9.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 scikit-learn==1.9.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'scikit-learn']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/recordings/asr/whisper-small.jsonl": "953f41bdba51fe75c411960af3757a8c545b29670c87ac9f04b25f824ff3cf6d",
    "data/recordings/asr/whisper-tiny.jsonl": "d6c12c960a2bc99981f866d674830b7501c01f754ae0f11d8bacf1139aa7285b",
    "data/recordings/llm/reason-reference.jsonl": "bef65a1ea9166d96941071bc80ba858d5e9b21c81a86a8be707f61e026ebc282",
    "data/recordings/llm/reason-whisper-small.jsonl": "82734affbfebd7d77900e0dfa126f5eaa951a011953c2e14690decdaeaf7b932",
    "compare.py": "99a57df2715cd87e50ce2b967afd530067dc626149f960b32d38f683d39c905f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The call reasons

Grace wants to know why people call: if a third of the calls ask "where is my order?", a better tracking email saves money. Every Larkfield call has a reason label (7 reasons). The **train** calls have transcripts only; the **test** calls also have audio and the recorded transcripts.

In [ ]:
calls = load_calls()
train = [c for c in calls.values() if c["split"] == "train"]
test = [c for c in calls.values() if c["split"] == "test"]
truth = [c["reason"] for c in test]
recognised = {}
for line in open("data/recordings/asr/whisper-small.jsonl", encoding="utf-8"):
    row = json.loads(line)
    if row["kind"] == "phone":
        recognised[row["call_id"]] = row["text"]
from collections import Counter
print(len(train), "train calls,", len(test), "test calls")
print(Counter(truth).most_common())

> **Check.** 150 train calls and 60 test calls. `order_status` is the most common reason.

## 2. The idea: keyword rules

The simplest method: a list of words for each reason, checked in a fixed order. The first reason with a word in the transcript wins. Run the next cell: it defines the rules and tries them on three sentences.

> **Predict.** The third sentence says "cancel" and "broken". Which reason do the rules give?

In [ ]:
# Keyword rules: the first reason whose words appear in the transcript wins (in this order).
RULES = [
    ("cancel_order", ["cancel", "annuler", "annulation"]),
    ("billing", ["charged", "charge", "twice", "payment", "card", "bill", "débité", "paiement", "facture", "carte",
                 "deux fois"]),
    ("damaged_item", ["damaged", "broken", "cracked", "abîmé", "cassé", "brisé", "endommagé", "fissuré"]),
    ("return_refund", ["return", "refund", "money back", "send it back", "retourner", "rembours", "retour"]),
    ("account_change", ["address", "account", "email", "adresse", "compte", "courriel"]),
    ("order_status", ["where", "arrived", "delivery", "tracking", "track", "où", "arrivé", "livraison", "suivi"]),
    ("product_question", ["how", "set up", "setup", "question", "comment", "installer", "installation"]),
]

def rule_reason(text, language="en"):
    t = " " + cp.normalise(text, "basic", language) + " "
    for reason, words in RULES:
        if any(f" {w}" in t for w in words):
            return reason
    return "product_question"

def accuracy(predicted):
    return float(np.mean([p == t for p, t in zip(predicted, truth)]))

for sentence in ["I was charged twice for my lantern.", "Where is my order? It has not arrived.",
                 "I want to cancel the hammock, it arrived broken."]:
    print(f"{rule_reason(sentence):16} <- {sentence}")

> **Check.** `cancel_order`: the rules check `cancel_order` first. The order of the rules decides when two reasons match. A person reads the whole sentence; the rules do not.

## 3. Three methods on the reference transcripts

First the rules on the exact transcripts (the reference).

In [ ]:
ref_text = [reference(c) for c in test]
rules_ref = [rule_reason(t, c["language"]) for t, c in zip(ref_text, test)]
print(f"rules on the reference transcripts: accuracy {accuracy(rules_ref):.3f}")

Next a **classifier**: TF-IDF turns each transcript into numbers (how often each word and word pair appears, compared with all calls), and logistic regression learns a weight for each one from the 150 train calls. The text classification course covers this in depth.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def train_classifier(texts, labels):
    """TF-IDF of words and word pairs, then logistic regression."""
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    model = LogisticRegression(max_iter=2000, C=10.0)
    model.fit(vec.fit_transform([cp.normalise(t, "basic") for t in texts]), labels)
    return lambda texts: list(model.predict(vec.transform([cp.normalise(t, "basic") for t in texts])))

classify = train_classifier([reference(c) for c in train], [c["reason"] for c in train])
clf_ref = classify(ref_text)
print(f"classifier on the reference transcripts: accuracy {accuracy(clf_ref):.3f}")

Last, an **LLM**: Azure OpenAI `chat-small` read each transcript and answered in JSON (reason, summary, promise, next step). The answers were recorded once, so you need no key. Each row has the cost and the answer time.

In [ ]:
def load_llm(name):
    rows = [json.loads(line) for line in open(f"data/recordings/llm/{name}.jsonl", encoding="utf-8") if line.strip()]
    return {r["call_id"]: r for r in rows}

llm_ref_rows = load_llm("reason-reference")
llm_ref = [llm_ref_rows[c["call_id"]]["answer"].get("reason") for c in test]
usd = sum(llm_ref_rows[c["call_id"]]["usd"] for c in test)
latency = np.mean([llm_ref_rows[c["call_id"]]["latency_s"] for c in test])
print(f"LLM on the reference transcripts: accuracy {accuracy(llm_ref):.3f}")
print(f"cost US${1000 * usd / len(test):.2f} for 1,000 calls, mean answer time {latency:.1f} s")
print(json.dumps(llm_ref_rows[test[0]["call_id"]]["answer"], indent=1, ensure_ascii=False))

> **Check.** On the reference transcripts: rules 0.9, classifier 0.967, LLM 1.0. The LLM costs about US$0.13 for 1,000 calls (at the deployment's price) and takes about 2.0 s for each answer; the rules and the classifier cost almost nothing.

## 4. Change one thing: English training calls only

Larkfield's calls are in English and French. What happens to French calls if the classifier learns from English calls only?

In [ ]:
english = [c for c in train if c["language"] == "en"]
classify_en = train_classifier([reference(c) for c in english], [c["reason"] for c in english])
pred_en = classify_en(ref_text)
for lang in ("en", "fr"):
    idx = [i for i, c in enumerate(test) if c["language"] == lang]
    print(lang, f"trained on both: {np.mean([clf_ref[i] == truth[i] for i in idx]):.3f}",
          f"  trained on English only: {np.mean([pred_en[i] == truth[i] for i in idx]):.3f}")

> **Check.** On French test calls: 0.923 when trained on both languages, 0.231 when trained on English only. A classifier only knows the words that it saw. The LLM reads both languages without training.

## 5. Failure case: testing on the training data

A common mistake: you train the classifier on the same calls that you test it on.

In [ ]:
leaky = train_classifier(ref_text, truth)          # trained on the test calls themselves
print(f"trained on the test calls: accuracy {accuracy(leaky(ref_text)):.3f} (meaningless)")

> **Check.** A near-perfect score that means nothing: the model remembers these calls. **Fix:** keep the test calls apart, and choose the method and its settings on the train (or validation) calls only.

## 6. Predict, then measure: from reference to recognised transcripts

In production there is no reference: each method reads Whisper's transcript, with its errors. Whisper `small` has the lowest WER of the recorded sizes.

> **Predict.** Which method loses the most accuracy when it reads the recognised transcript: the rules, the classifier or the LLM?

## 7. Your turn: the accuracy drop

> **Your turn.** Use the recorded Whisper `small` phone transcripts (`recognised[call_id]`) of the test calls. Compute the accuracy of the rules, of the classifier (`classify`, trained on the reference transcripts of the train calls) and of the recorded LLM answers on them (`load_llm("reason-whisper-small")`). Put the drop for each method (accuracy on the reference minus accuracy on the recognised transcripts) in a dictionary `drop` with the keys `"rules"`, `"classifier"` and `"llm"`.

In [ ]:
drop = ...

In [ ]:
expect_hash('drop', drop, 'f0aaa972b128d710')

> **Check.** Compare the drops with your prediction. On these calls, Whisper `small` (WER about 0.09 on the phone calls) gets the words that decide the reason right, so a method can lose nothing.

## 8. Change one thing: a weaker recogniser

Whisper `tiny` makes more than twice as many word errors as `small`. The LLM answers were recorded only for `small`, so the next cell runs the rules and the classifier on the recorded `tiny` transcripts.

> **Predict.** Which of the two loses more?

In [ ]:
tiny = {}
for line in open("data/recordings/asr/whisper-tiny.jsonl", encoding="utf-8"):
    row = json.loads(line)
    if row["kind"] == "phone":
        tiny[row["call_id"]] = row["text"]
tiny_text = [tiny[c["call_id"]] for c in test]
print(f"Whisper tiny transcripts: rules {accuracy([rule_reason(t, c['language']) for t, c in zip(tiny_text, test)]):.3f}, "
      f"classifier {accuracy(classify(tiny_text)):.3f}")

> **Check.** Rules 0.817 (from 0.9), classifier 0.95 (from 0.967). The rules depend on single words, so one wrong word changes the answer; the classifier uses all the words. Errors in the transcript move into every decision after it, and how much depends on the method.

## Next

You compared rules, a classifier and a recorded LLM for the call reason, and measured how recognition errors move the result. The [lesson page](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Summaries and follow-ups](https://learning.nextia-ai.com/courses/speech/m06/summaries-and-follow-ups/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m06/why-did-they-call/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.